# HDB Resale Analysis

In [ ]:
from pathlib import Path
import pandas as pd
import os
folder = Path("C:\\HDB_Project\\datasets")
output_folder = Path("C:\\HDB_Project\\datasets\\output")
start_date = '2012-01'
end_date = '2016-12'
def Extract_Files():
    # 1. Gather all CSV files in a specific directory
    file_paths = folder.glob('*.csv')

    # 2. Read and combine them into a single DataFrame
    df_list = [pd.read_csv(file) for file in file_paths]
    combined = pd.concat(df_list, ignore_index=True)
    #print(combined)

    # 3. Convert 'month' column to datetime and filter the range
    #combined['month'] = pd.to_datetime(combined['month'], format='%Y-%m')
    #print(combined)

    # Filter for January 2012 to December 2016
    filtered_df = combined[
        (combined['month'] >= start_date) &
        (combined['month'] <= end_date)
        ]

    combined = filtered_df

    # 4. Save the filtered records to a new CSV file
    # filtered_df.to_csv('filtered_records_2012_2016.csv', index=False)
    combined.to_csv(folder / "combined.csv", index=False)
    # To get the combined Records

    duplicates = combined[combined.duplicated(keep=False)]
    duplicates.to_csv(folder / "duplicate_records.csv",index=False)

    # To get the Cleaned Records without duplicates

    cleaned = combined.drop_duplicates()
    cleaned.to_csv(folder / "cleaned.csv",index=False)

    # Summary the Files
    original_count = len(combined)
    cleaned_count = len(cleaned)

    print("Original Rows  From Jan 2012 to Dec 2016 :", original_count)
    print("Original Rows  After removing Duplicate  :", cleaned_count)
    print("Duplicates Rows                          :", original_count - cleaned_count)

    return cleaned

def Data_Validation(cleaned):
    # =========================================
    # MONTH VALIDATION
    # =========================================
    report = []
    month_dates = pd.to_datetime(
        cleaned["month"],
        errors="coerce"
    )
    #print(month_dates)
    min_date = month_dates.min()
    max_date = month_dates.max()
    invalid_month = cleaned[
        month_dates.isna()
    ]

    report.append({
        "Column": "month",
        "Validation Rule": "Valid Date",
        "Invalid Records": len(invalid_month)
    })
    #print(report)

    # =========================================
    # TOWN VALIDATION
    # =========================================

    valid_towns = set(
        cleaned["town"]
        .dropna()
        .astype(str)
        .str.strip()
        .str.upper()
        .unique()
    )

    invalid_town = cleaned[
        (
            ~cleaned["town"]
            .astype(str)
            .str.upper()
            .str.match(
                r"^[A-Z\s]+$", #r"^[A-Z\s\-'/.&]+$"
                na=False
            )
        )
    ]

    report.append({
        "Column": "town",
        "Validation Rule":
            "Alphabetic + Master Dataset Values",
        "Invalid Records": len(invalid_town)
    })
    # ----------------------------------
    # FLAT TYPE VALIDATION
    # ----------------------------------

    invalid_flat_type = cleaned[
        ~cleaned["flat_type"]
        .astype(str)
        .str.match(
            r"^[A-Z0-9\s\-]+$",
            na=False
        )
    ]

    report.append({
        "Column": "flat_type",
        "Validation Rule":
            "Alphanumeric Characters",
        "Invalid Records":
            len(invalid_flat_type)
    })
    # ----------------------------------
    # FLAT MODEL VALIDATION
    # ----------------------------------

    invalid_model = cleaned[
        ~cleaned["flat_model"]
        .astype(str)
        .str.match(
            r"^[A-Za-z0-9\s\-]+$",
            na=False
        )
    ]

    report.append({
        "Column": "flat_model",
        "Validation Rule":
            "Alphanumeric Characters",
        "Invalid Records":
            len(invalid_model)
    })
    # ----------------------------------
    # STOREY RANGE VALIDATION
    # ----------------------------------

    invalid_storey = cleaned[
        ~cleaned["storey_range"]
        .astype(str)
        .str.match(
            r"^\d{2}\sTO\s\d{2}$",
            na=False
        )
    ]

    report.append({
        "Column": "storey_range",
        "Validation Rule":
            "Format NN TO NN",
        "Invalid Records":
            len(invalid_storey)
    })
    # ----------------------------------
    # CREATE REPORT
    # ----------------------------------

    validation_report = pd.DataFrame(report)

    validation_report.to_csv(
        folder / "validation_report.csv",
        index=False
    )

    invalid_records = pd.concat(
        [
            invalid_month,
            invalid_town,
            invalid_flat_type,
            invalid_model,
            invalid_storey
        ]
    )  #.drop_duplicates()

    invalid_records.to_csv(
        folder / "invalid_records.csv",
        index=False
    )

    print("\nValidation Report")
    print(validation_report)
    #print(report)

def Calculate_Lease_Period(cleaned):
    today = pd.Timestamp.today()

    # Convert lease commencement year to datetime
    lease_start = pd.to_datetime(
        cleaned["lease_commence_date"].astype(str),
        format="%Y",
        errors="coerce"
    )

    # Lease expiry date
    lease_expiry = (
        lease_start +
        pd.DateOffset(years=99)
    )

    # Calculate remaining months
    total_months = (
        (lease_expiry.dt.year - today.year) * 12
        +
        (lease_expiry.dt.month - today.month)
    )

    # Prevent negative values
    total_months = total_months.clip(lower=0)

    # Convert to years and months
    years = total_months // 12
    months = total_months % 12

    # Update remaining lease column
    cleaned["remaining_lease"] = (
        years.astype(int).astype(str)
        + " years "
        + months.astype(int).astype(str)
        + " months"
    )

    return cleaned
def Get_Max_Resale_Price(cleaned):

    # Composite key = all columns except resale_price
    key_cols = [c for c in cleaned.columns if c != "resale_price"]

    # Sort so highest resale price appears first within each key group
    sorted_df = cleaned.sort_values(
        by=key_cols + ["resale_price"],
        ascending=[True] * len(key_cols) + [False]
    )

    # Keep highest resale price record
    cleaned_final = sorted_df.drop_duplicates(
        subset=key_cols,
        keep="first"
    )

    # Lower-priced duplicate records go to failed dataset
    failed_duplicates = sorted_df[
        sorted_df.duplicated(subset=key_cols, keep="first")
    ].copy()
    cleaned.to_csv(folder / "cleaned_hdb_data.csv",index=False)
    return(cleaned)

def Resale_Price_Validation(cleaned,max_sale_price):
    cleaned_More_Value = cleaned[
            (cleaned["resale_price"] < 0) |
            (cleaned["resale_price"] >= max_sale_price)
    ].copy()
    return(cleaned_More_Value)

def Add_Resale_Identifier():
    import pandas as pd
    import re
    #import hashlib
    file_path = Path(folder) / "cleaned_hdb_data.csv"
    # load data
    df = pd.read_csv(file_path)

    # Step 1: Calculate average resale price by month, town, and flat_type
    avg_price = (
        df.groupby(['month', 'town', 'flat_type'])['resale_price']
        .mean()
        .reset_index(name='avg_resale_price')
    )

    avg_price['avg_resale_price'] = avg_price['avg_resale_price'].round(2)

    # Merge average price back to original dataframe
    df = df.merge(
        avg_price,
        on=['month', 'town', 'flat_type'],
        how='left'
    )

    # Step 2: Extract first 3 digits from block after removing non-numeric characters
    def get_block_digits(block):
        digits = re.sub(r'[^0-9]', '', str(block))
        return digits[:3].zfill(3)

    # Step 3: Get first 2 digits of average resale price
    def get_avg_digits(avg_price):
        return str(int(round(avg_price)))[:2].zfill(2)

    # Step 4: Get month digits (MM)
    df['month_digits'] = pd.to_datetime(df['month']).dt.strftime('%m')

    # Step 5: Create Resale Identifier
    df['Resale_Identifier'] = (
            'S'
            + df['block'].apply(get_block_digits)
            + df['avg_resale_price'].apply(get_avg_digits)
            + df['month_digits']
            + df['town'].str[0].str.upper()
    )

    # Display results
    #print(df[['month', 'town', 'flat_type', 'block', 'Resale_Identifier']].head())
    #print(df[['month', 'town', 'flat_type', 'block', 'Resale Identifier']])

     # Optional: Save output
    file_path = Path(folder) / "hdb_with_resale_identifier.csv"
    df.to_csv(file_path, index=False)

def Add_Hash_Value():
    import pandas as pd
    import hashlib

    # Read source file
    file_path = Path(folder) / "hdb_with_resale_identifier.csv"
    df = pd.read_csv(file_path)

    # Hash the identifier column
    df['Hash_Value_identifier'] = df['Resale_Identifier'].apply(
    lambda x: hashlib.sha256(str(x).encode('utf-8')).hexdigest()
    )

# Save output file
    file_path = Path(folder) / "hdb_with_resale_identifier_hashed.csv"
    df.to_csv(file_path, index=False)

    #print("Hashing completed.")

def Data_Profiling():

    import pandas as pd
    import sweetviz as sv
    file_path = Path(folder) / "cleaned_hdb_data.csv"
    df = pd.read_csv(file_path)
    report = sv.analyze(df)
    file_path = Path(folder) / "Data_Profiling_Report_HDB.html"
    report.show_html(file_path)

def Take_Backup():
    # To take Backup
    import shutil
    file_path = Path(folder) / "combined.csv"
    dest_file_path = Path(output_folder) / "combined.csv"
    shutil.copy(
        file_path,
        dest_file_path
    )
    if os.path.exists(file_path):
        os.remove(file_path)

    file_path = Path(folder) / "duplicate_records.csv"
    dest_file_path = Path(output_folder) / "duplicate_records.csv"
    shutil.copy(
        file_path,
        dest_file_path
    )
    if os.path.exists(file_path):
        os.remove(file_path)
    file_path = Path(folder) / "cleaned.csv"
    dest_file_path = Path(output_folder) / "cleaned.csv"
    shutil.copy(
        file_path,
        dest_file_path
    )
    if os.path.exists(file_path):
        os.remove(file_path)
    file_path = Path(folder) / "invalid_records.csv"
    dest_file_path = Path(output_folder) / "invalid_records.csv"
    shutil.copy(
        file_path,
        dest_file_path
    )
    if os.path.exists(file_path):
        os.remove(file_path)
    file_path = Path(folder) / "validation_report.csv"
    dest_file_path = Path(output_folder) / "validation_report.csv"
    shutil.copy(
        file_path,
        dest_file_path
    )
    if os.path.exists(file_path):
        os.remove(file_path)
    file_path = Path(folder) / "cleaned_hdb_data.csv"
    dest_file_path = Path(output_folder) / "cleaned_hdb_data.csv"
    shutil.copy(
        file_path,
        dest_file_path
    )
    if os.path.exists(file_path):
        os.remove(file_path)

    file_path = Path(folder) / "hdb_with_resale_identifier.csv"
    dest_file_path = Path(output_folder) / "hdb_with_resale_identifier.csv"
    shutil.copy(
        file_path,
        dest_file_path
    )
    if os.path.exists(file_path):
        os.remove(file_path)

    file_path = Path(folder) / "hdb_with_resale_identifier_hashed.csv"
    dest_file_path = Path(output_folder) / "hdb_with_resale_identifier_hashed.csv"
    shutil.copy(
        file_path,
        dest_file_path
    )
    if os.path.exists(file_path):
        os.remove(file_path)


# Main Program Calls
cleaned = Extract_Files()
cleaned_lease_period = Calculate_Lease_Period(cleaned)
Data_Validation(cleaned)
cleaned_Max_lease_price = Get_Max_Resale_Price(cleaned_lease_period)
Data_Profiling()
##cleaned_More_Value=Resale_Price_Validation(cleaned_Max_lease_price,1000000)
Add_Resale_Identifier()
Add_Hash_Value()
Take_Backup()

import nbformat as nbf

nb = nbf.v4.new_notebook()

nb.cells = [
    nbf.v4.new_markdown_cell("# HDB Resale Analysis"),
    nbf.v4.new_code_cell("import pandas as pd")
]
with open("C:\\HDB_Project\\HDB_assignment.ipynb", "w") as f:
    nbf.write(nb, f)

